# 03 · Đánh giá AEGIS: ablation, kiểm định ý nghĩa, phân tích trọng số gating

Thay cho các lệnh `select_fusion`, `ablation`, `significance`, `weights_analysis`, `export` trong README.
Mọi thứ được fit trên `val_gating_fit`; `test_benchmark` và các pool S4 chỉ được đọc **một lần** trong ablation.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image, display

from src.core.config import load_config
from src.core.contracts import SplitManifest

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 40)
cfg = load_config()
print("repo:", ROOT)

In [ ]:
# Bộ dữ liệu thật (GenImage đã chuẩn hoá). Nếu chưa có, notebook tự chuyển sang DEMO MODE:
# ảnh tổng hợp + embedding giả lập, chạy trên CPU trong ~1 phút. Số liệu DEMO không có ý nghĩa khoa học.
MANIFEST = ROOT / "data/normalized_v2/manifest.json"
DEMO = not MANIFEST.exists()
if DEMO:
    from src.models.demo_features import build_demo
    cfg, man = build_demo(ROOT / "data/demo", scale=6, cfg=cfg)
    MANIFEST = ROOT / "data/demo/images/manifest.json"
else:
    man = SplitManifest.load(MANIFEST)
print("DEMO MODE" if DEMO else "REAL DATA", "|", len(man.records), "records |", MANIFEST)

In [ ]:
HEAD_VARIANTS = ["clean", "aug"]
OUT = ROOT / ("reports/demo_v3" if DEMO else "reports/v3")
S4_SEEDS = [0, 1] if DEMO else [0, 1, 2, 3, 4]
B = 300 if DEMO else 2000
if DEMO:
    cfg["fewshot"]["n_values"] = [5, 20]     # demo S4 support pool is too small for N = 100
OUT.mkdir(parents=True, exist_ok=True)

## 1. Chọn kiểu fusion (chỉ dùng val_gating_fit / val_gating_verify)

In [ ]:
from src.evaluation import select_fusion
sel = select_fusion.run(cfg, HEAD_VARIANTS, OUT / "fusion_selection", manifest=MANIFEST, B=B)
print("fusion được chọn:", sel["selected"])
cfg.setdefault("gating", {})["fusion"] = sel["selected"]

## 2. Bảng ablation chính (S1 sạch · S2 JPEG · S3 blur/resize · S4 generator chưa thấy)

In [ ]:
from src.evaluation.ablation import run_all
rows = run_all(cfg, HEAD_VARIANTS, OUT, s4_seeds=S4_SEEDS, save_predictions=True)
abl = pd.DataFrame(rows)
full = abl[(abl.variant == "full") & (abl.level.isin(["clean", "all"])) & (abl.N.isna() | (abl.N == 20))]
display(full.pivot_table(index=["head_variant", "method"], columns=["scenario", "generator"], values="auroc"))

## 3. Kiểm định bootstrap ghép cặp (theo ảnh gốc): AEGIS so với từng baseline

In [ ]:
from src.evaluation import significance
sig = pd.DataFrame(significance.run(OUT / "predictions", OUT, B=B, manifest=MANIFEST, cfg=cfg))
pooled = sig[(sig.metric == "auroc") & sig.level.isin(["clean", "all"])]
display(pooled[["head_variant", "scenario", "generator", "N", "baseline", "delta", "ci_lo", "ci_hi", "significant"]]
        .head(40))

## 4. Trọng số gating có dịch chuyển theo mức suy giảm không?

In [ ]:
from src.evaluation import weights_analysis as wa
from src.evaluation.predictions import load_predictions
from src.learning.bundle_io import load_split
params = wa.load_degradation_params(MANIFEST)
wrows = []
for hv in HEAD_VARIANTS:
    tg = load_predictions(OUT / "predictions" / f"predictions_{hv}.npz")
    wrows += wa.analyse(cfg, hv, load_split(cfg, "val_gating_fit", hv), load_split(cfg, "val_gating_verify", hv),
                        params, tg)
wa.write_outputs(wrows, OUT)
for f in ("weights_by_blur.png", "weights_by_jpeg.png"):
    if (OUT / f).exists():
        display(Image(filename=str(OUT / f)))

## 5. Xuất hệ thống đã fit cho web demo (kiểm tra trên val_gating_verify, không đọc test)

In [ ]:
from src.learning.export import fit_system
system = fit_system(cfg, "aug")
path = ROOT / ("data/demo/serving/aegis_demo_aug.pt" if DEMO else "data/cache/serving/aegis_v3_aug.pt")
path.parent.mkdir(parents=True, exist_ok=True)
system.save(path)
print("saved", path, "| ood_threshold =", round(system.ood_threshold, 4))